In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder

load_dotenv()

C:\Users\abhay\AppData\Local\Temp\ipykernel_9844\595391706.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\abhay\Desktop\WEB dev\RAG\adv.RAG\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    dimensions=512
)

In [4]:
index_name = "learning-hybrid-search-index"

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name = index_name,
        dimension=512,
        metric='dotproduct'
    )

In [5]:
index = pc.Index(index_name)
index

Index(host='https://learning-hybrid-search-index-xia98fk.svc.aped-4627-b74a.pinecone.io')

In [6]:
docs = PyPDFLoader("../data/Company_Policies.pdf").load()+PyPDFLoader("../data/Company_Profile.pdf").load() + PyPDFLoader("../data/Product_and_pricing.pdf").load()
# docs = PyPDFLoader("../data/Company_Policies.pdf").load()

texts = [d.page_content for d in docs]

bm25 = BM25Encoder()
bm25.fit(texts)

100%|██████████| 8/8 [00:00<00:00, 57.88it/s]


In [7]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25,index=index)
retriever

PineconeHybridSearchRetriever(embeddings=OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002AE4A7171D0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002AE4A92CD10>, model='text-embedding-3-small', dimensions=512, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://openrouter.ai/api/v1', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000002AE4ACC7010>, index

In [8]:
retriever.add_texts(texts)

100%|██████████| 1/1 [00:05<00:00,  5.49s/it]


In [9]:
retriever.invoke("what is nexa ai refund date")

[Document(metadata={'score': 0.321123898}, page_content='NexaAI Solutions – Company Profile\nCompany Overview\nNexaAI Solutions Pvt. Ltd. is a business-focused artificial intelligence company founded in 2021.\nThe company specializes in building enterprise-ready AI systems for knowledge management,\nanalytics, and automation.\nNexaAI primarily serves mid-sized and large organizations across technology, finance, healthcare,\nand education sectors.\n\x7f\nFounded: 2021\n\x7f\nHeadquarters: Bengaluru, India\n\x7f\nEmployees: 85+\n\x7f\nOperating Regions: India, Southeast Asia, Europe\nVision\nTo become a trusted AI partner for organizations by enabling intelligent decision-making through\nreliable and explainable AI systems.\nMission\nOur mission is to simplify access to organizational knowledge, automate repetitive workflows, and\nhelp teams operate with speed, clarity, and confidence.\nWe focus on practical AI adoption rather than experimental research.\nCore Values\nOur values guide ho